In [ ]:
import os
os.environ["GROQ_API_KEY"] = "add you key here "
os.environ["EC_API_KEY"] = "add your key here"

In [8]:
!pip install langchain-groq
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00


In [9]:
# tool creation

from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_fact(base_currency : str , target_currency:str) -> float:
  """
  This fucntion fetches the currency conversion factor between a given base currency
  and a target requested currency
  """

  url = f"https://v6.exchangerate-api.com/v6/20fe1e87efd6ddc50f61c4c5/pair/{base_currency}/{target_currency}"
  response = requests.get(url)
  return response.json()


@tool
def convert(base_currency_value:int , conversion_rate:Annotated[float,InjectedToolArg]) -> float :
  """
    given a currency conversion rate this function calculates the target currency
    value from a given base currency value
  """

  return base_currency_value * conversion_rate



In [38]:
get_conversion_fact.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772150401,
 'time_last_update_utc': 'Fri, 27 Feb 2026 00:00:01 +0000',
 'time_next_update_unix': 1772236801,
 'time_next_update_utc': 'Sat, 28 Feb 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 91.027}

In [39]:
convert.invoke({'base_currency_value':'29' , 'conversion_rate':'90.965'})

'90.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.96590.965'

In [86]:
# tool binding

llm = ChatGroq(
    model = "llama-3.1-8b-instant"
)

In [87]:
llm_with_tools = llm.bind_tools([
    get_conversion_fact,
    convert
])

In [88]:
messages = [HumanMessage('what is the conversion factor between USD and INR and based on that can you convert 10 USD to INR')]

In [89]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [93]:
ai_message = llm_with_tools.invoke(messages)

In [92]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gvjc4zw4c', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_fact'}, 'type': 'function'}, {'id': 'v94xywk9w', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.0141}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 360, 'total_tokens': 408, 'completion_time': 0.085616206, 'completion_tokens_details': None, 'prompt_time': 0.029723338, 'prompt_tokens_details': None, 'queue_time': 0.069594877, 'total_time': 0.115339544}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8f8420ecd7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c9ee8-f2ed-76e2-aeda-f210ebe14156-0', tool_calls=[{'name': 'get_conversion_fact', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'gvjc4zw4c', 'type'

In [96]:
messages.append(ai_message)

In [97]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0ft421try', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_fact'}, 'type': 'function'}, {'id': '48kcs4c3z', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":82.92}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 360, 'total_tokens': 407, 'completion_time': 0.087589151, 'completion_tokens_details': None, 'prompt_time': 0.050302578, 'prompt_tokens_details': None, 'queue_time': 0.179536261, 'total_time': 0.137891729}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4848f70c04', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01

In [98]:
ai_message.tool_calls

[{'name': 'get_conversion_fact',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '0ft421try',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_factor': 82.92},
  'id': '48kcs4c3z',
  'type': 'tool_call'}]

In [104]:
import json

for tool_call in ai_message.tool_calls:
  #execution
  if tool_call ['name'] == 'get_conversion_fact':
    tool_message_1 = get_conversion_fact.invoke(tool_call)
    #fetch the coversion rate
    conversion_rate = (json.loads(tool_message_1.content)['conversion_rate'])
    #append this tool message to message list
  messages.append(tool_message_1)
  if tool_call['name'] == "convert":
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message_2 = convert.invoke(tool_call)
    messages.append(tool_message_2)

In [105]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0ft421try', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_fact'}, 'type': 'function'}, {'id': '48kcs4c3z', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":82.92}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 360, 'total_tokens': 407, 'completion_time': 0.087589151, 'completion_tokens_details': None, 'prompt_time': 0.050302578, 'prompt_tokens_details': None, 'queue_time': 0.179536261, 'total_time': 0.137891729}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4848f70c04', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01

[HumanMessage(content='what is the conversion factor between USD and INR and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0ft421try', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_fact'}, 'type': 'function'}, {'id': '48kcs4c3z', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":82.92}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 360, 'total_tokens': 407, 'completion_time': 0.087589151, 'completion_tokens_details': None, 'prompt_time': 0.050302578, 'prompt_tokens_details': None, 'queue_time': 0.179536261, 'total_time': 0.137891729}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4848f70c04', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01